# Where, not just what — keeping the patch axis

Every image notebook so far has done the same thing at the end: turn a picture into **one vector**,
then predict **one label**. That is the right shape for "is this a flower". It is the wrong shape for
"*where* are the trees".

This notebook removes one line — the aggregation — and gets a map instead of a label.

## The rung you already know

Lesson 1 introduced the tensor ladder, and lesson 5 spent a whole notebook on `(B, C, D)`: a batch of
`B` documents, each cut into `C` chunks, each chunk a `D`-dimensional vector. Aggregating `C` away
gave one vector per document.

**An image is the same rung.** A vision transformer cuts a picture into patches and produces one
vector per patch. `C` is no longer "chunks of text", it is "patches of image" — and everything the
course already taught about that axis applies unchanged.

| | `B` | `C` | `D` |
|---|---|---|---|
| lesson 5, SCOTUS | documents | 512-token chunks | 384 |
| here, EuroSAT | tiles | 14x14-pixel patches | 384 |

Aggregate `C` -> classification. Keep `C` -> segmentation. That is the whole idea.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torchvision.transforms import v2

torch.manual_seed(0)
RES = 224            # what the pretrained encoders expect
N_TRAIN, N_TEST = 300, 100

## 1. What pooling throws away

Load a pretrained encoder and look at what it actually computes, before anything reduces it.

In [ ]:
from PIL import Image

from vectormesh import ImageVectorizer, PatchImageVectorizer

MODEL = "facebook/dinov2-small"
demo = [Image.new("RGB", (RES, RES))]

pooled_vec = ImageVectorizer(model_name=MODEL)        # one vector per image
patch_vec = PatchImageVectorizer(model_name=MODEL)    # one vector per patch

pooled = pooled_vec(demo, batchsize=1)["embed"][0]
patches = patch_vec(demo, batchsize=1)["patches"][0]
grid = patch_vec.get_grid
assert grid is not None

print(f"ImageVectorizer      -> {str(tuple(pooled.shape)):>12}   one vector for the whole image")
print(f"PatchImageVectorizer -> {str(tuple(patches.shape)):>12}   one vector per patch")
print()
print(f"patch grid: {grid[0]}x{grid[1]}  "
      f"= {patches.shape[0]} vectors, all but one of which pooling throws away.")

`(1, 257, 384)` is **1 summary token + 256 patch tokens**. The 224x224 image is cut into a 16x16 grid
of 14-pixel patches, and the model produces a 384-dim vector for each one. Those vectors know *where*
they are.

`vectormesh`'s `ImageVectorizer` returns the pooled `(1, 384)`. For classification that is exactly
right — and it is why the caches from lesson 2 are so small. But it is a lossy choice, and this
notebook is about when to make the other one.

## 2. A segmentation task, built from a classification dataset

EuroSAT has **no masks** — one label per 64x64 tile, nothing about *where* within the tile.

So we build the labels we need: paste four tiles of known classes into a 2x2 composite. Now we know
the ground truth for every region by construction, and we can ask whether an encoder recovers it.

This is a deliberately easy segmentation problem — the boundaries are straight lines in known places.
That is the point: if patch tokens cannot solve *this*, they will not solve a real one, and we find
out in five seconds instead of after building a whole pipeline.

In [ ]:
from datasets import load_dataset

ds = load_dataset("blanchon/EuroSAT_RGB")
CLASSES = ds["train"].features["label"].names
tiles_raw = ds["train"].shuffle(seed=0)

half = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
                   v2.Resize((RES // 2, RES // 2), antialias=True)])

t0 = time.time()
pool_tiles = [(half(tiles_raw[i]["image"].convert("RGB")), tiles_raw[i]["label"])
              for i in range(4000)]
print(f"{len(pool_tiles)} tiles ready in {time.time()-t0:.0f}s")


def make_composite(rng):
    """2x2 grid of random tiles -> (3, RES, RES) image + its 4 quadrant labels (TL TR BL BR)."""
    picks = [pool_tiles[rng.randint(0, len(pool_tiles) - 1)] for _ in range(4)]
    top = torch.cat([picks[0][0], picks[1][0]], dim=2)
    bottom = torch.cat([picks[2][0], picks[3][0]], dim=2)
    return torch.cat([top, bottom], dim=1), torch.tensor([p[1] for p in picks])


import random
rng = random.Random(0)
images, quadrants = zip(*[make_composite(rng) for _ in range(N_TRAIN + N_TEST)])
X = torch.stack(images)
Q = torch.stack(quadrants)
print(f"composites {tuple(X.shape)}, quadrant labels {tuple(Q.shape)}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, k in zip(axes, range(4)):
    ax.imshow(X[k].permute(1, 2, 0))
    ax.axhline(RES // 2, color="w", lw=1); ax.axvline(RES // 2, color="w", lw=1)
    ax.set_title(" | ".join(CLASSES[c][:11] for c in Q[k][:2]) + "\n"
                 + " | ".join(CLASSES[c][:11] for c in Q[k][2:]), fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Embed twice: once pooled, once not

Same encoder, same forward pass. The only difference is what we keep.

In [ ]:
# The vectorizer takes PIL images and handles its own normalisation, so hand it
# the composites directly rather than pre-processing them here.
as_pil = [v2.functional.to_pil_image(img) for img in X]

t0 = time.time()
T = torch.stack(patch_vec(as_pil, batchsize=16)["patches"])   # (N, 256, 384)
GRID = int(T.shape[1] ** 0.5)
P = T.mean(dim=1)                             # (N, 384) - the pooled version, for comparison
print(f"patch tokens {tuple(T.shape)} -> {GRID}x{GRID} grid   [{time.time()-t0:.0f}s]")
print(f"pooled       {tuple(P.shape)}")

In [ ]:
# Which quadrant does each patch fall into? Pure geometry - no model involved.
rows = torch.arange(GRID).view(GRID, 1).expand(GRID, GRID)
cols = torch.arange(GRID).view(1, GRID).expand(GRID, GRID)
quad_of_patch = ((rows >= GRID // 2).long() * 2 + (cols >= GRID // 2).long()).reshape(-1)

Y = Q[:, quad_of_patch]        # (N, 256) - a label per patch
print(f"per-patch labels {tuple(Y.shape)}")
print(f"patch 0 sits in quadrant {quad_of_patch[0]}, patch 255 in quadrant {quad_of_patch[-1]}")

## 4. The comparison

Two probes, one frozen encoder:

- **pooled** — one 384-dim vector per composite. We ask it for each quadrant's label in turn.
- **per-patch** — 256 vectors per composite, each with its own label.

In [ ]:
def probe(Xtr, ytr, Xte, yte, epochs=300, lr=1e-2):
    mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True) + 1e-6
    Xtr_, Xte_ = (Xtr - mu) / sd, (Xte - mu) / sd
    clf = nn.Linear(Xtr.shape[1], len(CLASSES))
    opt = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-4)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(clf(Xtr_), ytr).backward(); opt.step()
    with torch.no_grad():
        acc = (clf(Xte_).argmax(1) == yte).float().mean().item()
    return acc, clf, (mu, sd)


ntr = N_TRAIN
pooled_accs = [probe(P[:ntr], Q[:ntr, q], P[ntr:], Q[ntr:, q])[0] for q in range(4)]

patch_acc, patch_clf, patch_norm = probe(
    T[:ntr].reshape(-1, T.shape[-1]), Y[:ntr].reshape(-1),
    T[ntr:].reshape(-1, T.shape[-1]), Y[ntr:].reshape(-1),
)

print(f"pooled vector -> quadrant labels : {[f'{a:.3f}' for a in pooled_accs]}"
      f"   mean {sum(pooled_accs)/4:.3f}")
print(f"per-patch probe (no aggregation) : {patch_acc:.3f}")

The pooled vector scores about the same on every quadrant, and that number is low. This is not the
encoder being weak — it is a **shape problem**. One vector was asked to describe a scene containing
four different land types; it has no way to say which one is in the top left.

Removing the aggregation roughly doubles the score, with the same encoder and the same frozen weights.

In [ ]:
def predict_map(clf, norm, tokens):
    mu, sd = norm
    with torch.no_grad():
        return clf((tokens - mu) / sd).argmax(1).reshape(GRID, GRID)


fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for k in range(4):
    idx = ntr + k
    truth = Q[idx][quad_of_patch].reshape(GRID, GRID)
    pred = predict_map(patch_clf, patch_norm, T[idx])
    axes[0, k].imshow(X[idx].permute(1, 2, 0)); axes[0, k].set_title("composite", fontsize=9)
    axes[1, k].imshow(pred, cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
    axes[1, k].set_title(f"predicted map ({(pred == truth).float().mean():.0%} correct)", fontsize=9)
    for ax in (axes[0, k], axes[1, k]):
        ax.axis("off")
plt.tight_layout(); plt.show()

Two honest observations about those maps:

1. **They are blocky.** Each cell is a 14x14-pixel patch, so the finest boundary this can express is
   14 pixels wide. Getting to pixel-level masks needs a decoder that upsamples — a different notebook.
   Per-patch prediction buys most of the value for a fraction of the machinery.
2. **The seams are one patch thick.** Patches straddling the quadrant boundary genuinely contain two
   land types, so the model is not wrong to be unsure there. Errors concentrate exactly where the
   ambiguity is real, which is a good sign rather than a bad one.

## 5. The CNN has a spatial grid too

It is easy to think this is a transformer trick. It is not. A convolutional network keeps a spatial
grid all the way through — `AdaptiveAvgPool2d(1)`, the layer just before the classifier, is precisely
where that grid gets thrown away.

In [ ]:
import torchvision

# the ResNets below are raw state dicts, so they take the manual path (see note);
# reuse the encoder's own normalisation constants for a like-for-like comparison.
_proc = patch_vec.get_processor
mean = torch.tensor(_proc.image_mean).view(1, 3, 1, 1)
std = torch.tensor(_proc.image_std).view(1, 3, 1, 1)

resnet = torchvision.models.resnet50()
backbone = nn.Sequential(*list(resnet.children())[:-2])   # everything except avgpool + fc
with torch.no_grad():
    feat = backbone(torch.zeros(1, 3, RES, RES))
print(f"resnet50 before pooling : {tuple(feat.shape)}   <- (batch, dim, h, w), a {feat.shape[-2]}x{feat.shape[-1]} grid")
print(f"after AdaptiveAvgPool2d : {tuple(F.adaptive_avg_pool2d(feat, 1).flatten(1).shape)}")

Same story, coarser grid: 7x7 cells of 2048 dims instead of 16x16 of 384. The `(B, C, D)` view is
obtained by flattening the spatial dims — `(b, 2048, 7, 7) -> (b, 49, 2048)`.

## 6. Which encoder should you use for aerial imagery?

Three frozen encoders, same probe:

- **DINOv2-small** — self-supervised on 142M general web images. What lesson 2 uses.
- **MoCo / Sentinel-2** — a ResNet-50 trained with **MoCo** (the method from notebook 01) on Sentinel-2
  satellite imagery. EuroSAT *is* Sentinel-2, so this encoder saw this sensor during pretraining.
- **ResNet-50 / ImageNet** — supervised, general. The control that isolates what the domain buys.

> **Why the ResNets below do not use `PatchImageVectorizer`.** The vectorizer wraps a model
> *by hub name*, which covers DINOv2. The two ResNets are raw `state_dict`s loaded into a
> `torchvision` architecture — there is no hub id to hand over — so they take the manual
> route. Worth knowing where a library's reach ends, rather than discovering it mid-task.

> **The D16 habit applies here.** `torchgeo`'s MoCo was pretrained on Sentinel-2, and EuroSAT is
> Sentinel-2 — so a strong score is partly a *domain* result and possibly partly a *memorisation*
> result. Check the pretraining corpus before you quote a number.

In [ ]:
from huggingface_hub import hf_hub_download


def resnet_grid_embeddings(state_dict=None, images=X, batch=16):
    """ResNet-50 -> (N, 49, 2048): flatten the spatial grid instead of pooling it."""
    m = torchvision.models.resnet50()
    if state_dict is not None:
        m.load_state_dict(state_dict, strict=False)
    body = nn.Sequential(*list(m.children())[:-2]).eval()
    chunks = []
    with torch.no_grad():
        for i in range(0, len(images), batch):
            f = body((images[i:i + batch] - mean) / std)      # (b, 2048, 7, 7)
            chunks.append(f.flatten(2).transpose(1, 2))       # (b, 49, 2048)
    return torch.cat(chunks)


# MoCo pretrained on Sentinel-2. It is a plain resnet50 state dict, so no torchgeo needed.
moco_sd = torch.load(
    hf_hub_download("torchgeo/resnet50_sentinel2_rgb_moco", "resnet50_sentinel2_rgb_moco.pth"),
    map_location="cpu", weights_only=True,
)
moco_sd = moco_sd.get("state_dict", moco_sd)

imagenet_sd = torchvision.models.ResNet50_Weights.IMAGENET1K_V2.get_state_dict()

t0 = time.time()
grids = {
    "MoCo / Sentinel-2": resnet_grid_embeddings(moco_sd),
    "ResNet50 / ImageNet": resnet_grid_embeddings(imagenet_sd),
}
print(f"embedded both resnets in {time.time()-t0:.0f}s")

In [ ]:
def grid_labels(grid):
    """Quadrant per cell, plus a mask of cells that lie ENTIRELY inside one quadrant.

    A GxG grid over a 2x2 composite only splits cleanly when G is even. For G=7 the
    middle row and column straddle the midline, so 13 of 49 cells would be handed a
    label that is wrong for half their pixels. Scoring on those measures our grid,
    not the encoder -- so they are excluded from every encoder equally.
    """
    idx = torch.arange(grid)
    lo, hi = idx / grid, (idx + 1) / grid
    clean = ~((lo < 0.5) & (hi > 0.5))                       # cell does not span the midline
    rows = idx.view(grid, 1).expand(grid, grid)
    cols = idx.view(1, grid).expand(grid, grid)
    quad = ((rows >= grid / 2).long() * 2 + (cols >= grid / 2).long()).reshape(-1)
    keep = (clean.view(grid, 1) & clean.view(1, grid)).reshape(-1)
    return quad, keep


results = {"DINOv2-small (ViT)": (T, GRID)}
results.update({k: (g, int(g.shape[1] ** 0.5)) for k, g in grids.items()})

print(f"{'encoder':22s} {'grid':>7s} {'cells':>7s} {'dim':>6s} {'per-cell':>9s} {'pooled':>8s}")
print("-" * 64)
for name, (tokens, grid) in results.items():
    q_of_p, keep = grid_labels(grid)
    tok = tokens[:, keep, :]
    Yg = Q[:, q_of_p[keep]]
    acc, _, _ = probe(tok[:ntr].reshape(-1, tok.shape[-1]), Yg[:ntr].reshape(-1),
                      tok[ntr:].reshape(-1, tok.shape[-1]), Yg[ntr:].reshape(-1))
    pooled = tokens.mean(1)
    pacc = sum(probe(pooled[:ntr], Q[:ntr, q], pooled[ntr:], Q[ntr:, q])[0] for q in range(4)) / 4
    print(f"{name:22s} {grid:>3d}x{grid:<3d} {int(keep.sum()):>4d}/{grid*grid:<3d} "
          f"{tokens.shape[-1]:6d} {acc:9.3f} {pacc:8.3f}")

### Making that comparison fair

The first version of this table was **wrong**, in a way worth keeping visible.

A 2x2 composite splits at the midline. A 16x16 grid has a cell boundary exactly there, so every patch
sits cleanly in one quadrant. A **7x7 grid does not**: cell 3 spans [0.429, 0.571), straddling the
midline, so the whole middle row and column — **13 of 49 cells, 27%** — get a label that is wrong for
half of their pixels. The ResNets were being scored on coin flips that DINOv2 never faced.

`grid_labels` therefore drops straddling cells for every encoder equally, leaving 6x6=36 clean cells
for the ResNets and all 256 for DINOv2.

What survives is still not a *clean* encoder comparison — 36 large cells is an easier localisation
problem than 256 small ones, and dimensionality differs too. The two ResNets are directly comparable
to each other; DINOv2 is in a different column of the experiment. Say that, rather than declaring a
winner across rows.

## 7. What this means for a real aerial task

Everything above is a rehearsal. On a real dataset with real masks the pipeline is unchanged:

1. run the encoder once over every tile, keeping the patch axis, and cache `(C, D)` per tile
2. take per-patch labels by downsampling the mask to the patch grid
3. train a small head on the frozen patches — seconds on a laptop, because the expensive part already
   happened
4. upsample the predicted map back to image resolution if you need pixel-level output

Step 1 is the only part that needs a GPU, and it happens once — the `embed once, reuse many` principle
from lesson 1, applied to a task that looks nothing like what lesson 1 was doing.

## What would move into vectormesh

`ImageVectorizer` always pools. The change is one argument:

```python
ImageVectorizer(model_name="facebook/dinov2-small", pool="none")   # -> (C, D) per image
```

which makes an image cache the same shape as a text cache, so `FixedPadding`, the aggregators and
`Collate` all work on it unchanged. The aggregator you already know then becomes the switch between
the two tasks: **aggregate for classification, skip it for segmentation.**

## Exercises

**E1 — Where does it fail?** Build composites from *two* classes instead of four, choosing a pair the
maps confused above (Industrial vs Highway is a good start). Does the per-patch probe still separate
them, or was it relying on the other quadrants for context?

**E2 — How coarse can you go?** Downsample the patch grid by averaging 2x2 blocks of patch tokens
(16x16 -> 8x8) and re-probe. How much resolution can you give up before the map stops being useful?
This is the classification/segmentation trade-off as a dial rather than a binary.

**E3 — Does the domain matter more than the method?** The table has MoCo/Sentinel-2 and
ResNet50/ImageNet at the same grid and the same dimension, so that comparison is clean. Which wins,
and how would you tell a *domain* advantage apart from a *pretraining-corpus overlap* advantage?

**E4 — 3x3 composites.** Nine quadrants on a 16x16 grid means boundaries that no longer align with
patch edges. Predict what happens to the seams before you run it.

**E5 — The honest baseline.** Train the same per-patch probe on *raw pixel patches* (reshape the image
into 16x16 blocks of 14x14x3 = 588 values). How much of the score came from the pretrained encoder,
and how much from simply looking at a small region?

### What the table actually says

The two ResNets share a grid, a cell count and a dimensionality, so that row is a fair fight:

**ResNet50/ImageNet 0.769 vs MoCo/Sentinel-2 0.507.** The general-purpose *supervised* encoder beats
the domain-matched *self-supervised* one by 26 points — on the very sensor MoCo was pretrained on.

That is the opposite of what "use the domain-specific model" would predict, and the likely reason is
notebook 01 arriving from the other direction. **MoCo is trained to make two crops of the same tile
produce the same vector.** It is optimised to *discard* where-in-the-image information, because that
is exactly what differs between two crops. The invariance that makes it good at "what is in this
tile" is what makes it bad at "what is in this patch". This is why dense-prediction variants of
contrastive learning — DenseCL, PixPro — exist at all.

**But do not generalise it to "self-supervised learning is bad for segmentation."** DINOv2 is also
self-supervised, and it localises fine. The difference is the objective: DINOv2 (via iBOT) includes a
**patch-level** masked-prediction term, so patches are trained to mean something individually. MoCo
has only an image-level term.

So the rule is not about supervised versus self-supervised. It is:

> **The level your objective operates at is the level your features will be good at.**
> Image-level objective -> image-level features. Want per-patch features? Use a per-patch objective.

Which is also the argument for the JEPA-style masked prediction in notebook 01: predicting *a
particular masked block* is a patch-level objective, and that is what makes it a better fit for dense
tasks than image-level contrastive learning.

Hold this as a well-supported hypothesis rather than a proven claim — one dataset, one composite
geometry, two checkpoints. E3 asks you to push on it.